# Padding y stride

**Capítulo 4 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_convolutional-neural-networks/padding-and-strides.ipynb` · [Lección original](https://d2l.ai/chapter_convolutional-neural-networks/padding-and-strides.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Padding y stride
<a id="sec_padding"></a>

Recordemos el ejemplo de una convolución en [Referencia fig_correlation](https://d2l.ai/chapter_convolutional-neural-networks/conv-layer.html#fig-correlation). La entrada tenía una altura y una anchura de 3 y el núcleo de convolución tenía una altura y una anchura de 2, dando una representación de salida con dimensión $2\times2$. Suponiendo que la forma de entrada es $n_\textrm{h}\times n_\textrm{w}$ y la forma del núcleo de convolución es $k_\textrm{h}\times k_\textrm{w}$, la forma de salida será $(n_\textrm{h}-k_\textrm{h}+1) \times (n_\textrm{w}-k_\textrm{w}+1)$: sólo podemos cambiar el núcleo de convolución hasta que se quede sin píxeles para aplicar la convolución a.

En lo siguiente exploraremos una serie de técnicas, incluyendo relleno y convoluciones con stride, que ofrecen más control sobre el tamaño de la salida. Como motivación, tenga en cuenta que como los núcleos generalmente tienen anchura y altura mayores que $1$, después de aplicar muchas convoluciones sucesivas, tendemos a terminar con salidas considerablemente más pequeñas que nuestra entrada. Si empezamos con una imagen de píxeles $240 \times 240$, diez capas de convoluciones $5 \times 5$ reducen la imagen a píxeles $200 \times 200$, cortando $30 \%$ de la imagen y con ella borrando cualquier información interesante sobre los límites de la imagen original. *Padding* es la herramienta más popular para manejar este problema. En otros casos, podemos querer reducir drásticamente la dimensión, por ejemplo, si encontramos que la resolución original de entrada es inviable. *Las convoluciones con stride* son una técnica popular que puede ayudar en estos casos.


In [ ]:
import torch
from torch import nn

## Relleno
Como se ha descrito anteriormente, un problema difícil al aplicar capas convolucionales es que tendemos a perder píxeles en el perímetro de nuestra imagen. Considere [Referencia img_conv_reuse](https://d2l.ai/chapter_convolutional-neural-networks/padding-and-strides.html#img-conv-reuse) que representa la utilización de píxeles como una función del tamaño del núcleo de convolución y la posición dentro de la imagen. Los píxeles en las esquinas apenas se utilizan en absoluto.

![Uso de píxeles con convoluciones de tamaño $1 \times 1$, $2 \times 2$ y $3 \times 3$.](../recursos/originales/conv-reuse.svg)
<a id="img_conv_reuse"></a>

Ya que normalmente usamos pequeños núcleos, para cualquier convolución dada podemos perder sólo unos pocos píxeles, pero esto puede sumar como aplicamos muchas capas convolucionales sucesivas. Una solución directa a este problema es añadir píxeles extra de relleno alrededor del límite de nuestra imagen de entrada, aumentando así el tamaño efectivo de la imagen. Típicamente, establecemos los valores de los píxeles extra a cero. En [Referencia img_conv_pad](https://d2l.ai/chapter_convolutional-neural-networks/padding-and-strides.html#img-conv-pad), pad una entrada $3 \times 3$, aumentando su tamaño a $5 \times 5$. La salida correspondiente entonces aumenta a una matriz $4 \times 4$. Las porciones sombreadas son el primer elemento de salida, así como los elementos tensores de entrada y núcleo utilizados para el cálculo de salida: $0\times0+0\times1+0\times2+0\times3=0$.

![Correlación cruzada bidimensional con padding.](../recursos/originales/conv-pad.svg)
<a id="img_conv_pad"></a>

En general, si añadimos un total de $p_\textrm{h}$ filas de relleno (aproximadamente la mitad en la parte superior y la mitad en la parte inferior) y un total de $p_\textrm{w}$ columnas de relleno (aproximadamente la mitad a la izquierda y la mitad a la derecha), la forma de salida será

$$(n_\textrm{h}-k_\textrm{h}+p_\textrm{h}+1)\times(n_\textrm{w}-k_\textrm{w}+p_\textrm{w}+1).$$

Esto significa que la altura y el ancho de la salida aumentará en $p_\textrm{h}$ y $p_\textrm{w}$, respectivamente.

En muchos casos, queremos establecer $p_\textrm{h}=k_\textrm{h}-1$ y $p_\textrm{w}=k_\textrm{w}-1$ para dar a la entrada y salida la misma altura y anchura. Esto hará más fácil predecir la forma de salida de cada capa al construir la red. Suponiendo que $k_\textrm{h}$ es impar aquí, almohadillaremos las filas $p_\textrm{h}/2$ en ambos lados de la altura. Si $k_\textrm{h}$ es par, una posibilidad es almohadillar las filas $\lceil p_\textrm{h}/2\rceil$ en la parte superior de la entrada y $\lfloor p_\textrm{h}/2\rfloor$ en la parte inferior. Almohadillaremos ambos lados de la anchura de la misma manera.

Las CNN comúnmente usan núcleos de convolución con valores impares de altura y anchura, como 1, 3, 5, o 7. Elegir tamaños de núcleo impares tiene el beneficio de que podemos preservar la dimensionalidad mientras que el relleno con el mismo número de filas en la parte superior e inferior, y el mismo número de columnas en la izquierda y la derecha.

Además, esta práctica de usar núcleos impares y rellenos para preservar con precisión la dimensionalidad ofrece un beneficio clerical. Para cualquier tensor bidimensional `X`, cuando el tamaño del núcleo es impar y el número de filas y columnas de relleno en todos los lados es el mismo, produciendo así una salida con la misma altura y anchura que la entrada, sabemos que la salida `Y[i, j]` se calcula mediante la correlación cruzada del núcleo de entrada y convolución con la ventana centrada en `X[i, j]`.

En el siguiente ejemplo, creamos una capa convolucional bidimensional con una altura y anchura de 3 y **aplicar 1 píxel de relleno en todos los lados.** Dado una entrada con una altura y anchura de 8, encontramos que la altura y anchura de la salida es también 8.


In [ ]:
# Definimos una función de ayudante para calcular las convoluciones.
# la capa convolucional pesa y realiza la dimensión correspondiente
# elevaciones y reducciones de la entrada y la salida
def comp_conv2d(conv2d, X):
    # (1, 1) indica que el tamaño del lote y el número de canales son ambos 1
    X = X.reshape((1, 1) + X.shape)
    Y = conv2d(X)
    # Quitar las dos primeras dimensiones: ejemplos y canales
    return Y.reshape(Y.shape[2:])

# 1 fila y columna está rellenada en cada lado, por lo que un total de 2 filas o columnas
# se añaden
conv2d = nn.LazyConv2d(1, kernel_size=3, padding=1)
X = torch.rand(size=(8, 8))
comp_conv2d(conv2d, X).shape

Cuando la altura y el ancho del núcleo de la convolución son diferentes, podemos hacer que la salida y la entrada tengan la misma altura y anchura ** fijando diferentes números de relleno para la altura y la anchura.**


In [ ]:
# Utilizamos un núcleo de convolución con altura 5 y anchura 3.
# lado de la altura y el ancho son 2 y 1, respectivamente
conv2d = nn.LazyConv2d(1, kernel_size=(5, 3), padding=(2, 1))
comp_conv2d(conv2d, X).shape

### Nota docente de Hespérides

Comprueba primero las formas y el supuesto arquitectónico: localidad, compartición de pesos o conexión residual. El explorador permite seguir ventana, multiplicaciones y suma. En PyTorch, Conv2d implementa correlación cruzada; en aprendizaje profundo se suele llamar convolución a esta operación. El autoencoder 91 amplía el patrón MLP con reconstrucción; VAE se trata como contraste conceptual, al no existir un original válido en las fuentes locales.

Vínculo con los apuntes: sesión 4, «Padding y stride».


## Stride
Cuando computamos la correlación cruzada, empezamos con la ventana de convolución en la esquina superior izquierda del tensor de entrada, y luego la deslizamos por todas las ubicaciones hacia abajo y hacia la derecha. En los ejemplos anteriores, por defecto deslizamos un elemento a la vez. Sin embargo, a veces, ya sea para la eficiencia computacional o porque deseamos reducir la muestra, movemos nuestra ventana más de un elemento a la vez, omitiendo las ubicaciones intermedias. Esto es particularmente útil si el núcleo de convolución es grande ya que captura un área grande de la imagen subyacente.

Nos referimos al número de filas y columnas atravesadas por diapositiva como *stride*. Hasta ahora, hemos utilizado pasos de 1, tanto para la altura como para la anchura. A veces, podemos querer utilizar un paso más grande.
[Referencia img_conv_stride](https://d2l.ai/chapter_convolutional-neural-networks/padding-and-strides.html#img-conv-stride) muestra una operación de correlación cruzada bidimensional
Las porciones sombreadas son los elementos de salida, así como los elementos de tensor de entrada y núcleo utilizados para el cálculo de salida: $0\times0+0\times1+1\times2+2\times3=8$, $0\times0+6\times1+0\times2+0\times3=6$. Podemos ver que cuando se genera el segundo elemento de la primera columna, la ventana de convolución se desliza por tres filas. La ventana de convolución desliza dos columnas a la derecha cuando se genera el segundo elemento de la primera fila. Cuando la ventana de convolución continúa deslizando dos columnas a la derecha en la entrada, no hay salida porque el elemento de entrada no puede llenar la ventana (a menos que añadamos otra columna de relleno).

![Correlación cruzada con strides de 3 en altura y 2 en anchura.](../recursos/originales/conv-stride.svg)
<a id="img_conv_stride"></a>

En general, cuando el paso para la altura es $s_\textrm{h}$ y el paso para la anchura es $s_\textrm{w}$, la forma de salida es

$$\lfloor(n_\textrm{h}-k_\textrm{h}+p_\textrm{h}+s_\textrm{h})/s_\textrm{h}\rfloor \times \lfloor(n_\textrm{w}-k_\textrm{w}+p_\textrm{w}+s_\textrm{w})/s_\textrm{w}\rfloor.$$

Si establecemos $p_\textrm{h}=k_\textrm{h}-1$ y $p_\textrm{w}=k_\textrm{w}-1$, entonces la forma de salida se puede simplificar a $\lfloor(n_\textrm{h}+s_\textrm{h}-1)/s_\textrm{h}\rfloor \times \lfloor(n_\textrm{w}+s_\textrm{w}-1)/s_\textrm{w}\rfloor$. Ir un paso más allá, si la altura de entrada y el ancho son divisibles por los pasos en la altura y el ancho, entonces la forma de salida será $(n_\textrm{h}/s_\textrm{h}) \times (n_\textrm{w}/s_\textrm{w})$.

Abajo, ** establecemos los pasos tanto en la altura como en la anchura a 2**, así reduciendo a la mitad la altura de entrada y la anchura.


In [ ]:
conv2d = nn.LazyConv2d(1, kernel_size=3, padding=1, stride=2)
comp_conv2d(conv2d, X).shape

Echemos un vistazo a **un ejemplo un poco más complicado**.


In [ ]:
conv2d = nn.LazyConv2d(1, kernel_size=(3, 5), padding=(0, 1), stride=(3, 4))
comp_conv2d(conv2d, X).shape

## Resumen y debate
El padding puede aumentar la altura y el ancho de la salida. Esto se utiliza a menudo para dar a la salida la misma altura y el ancho de la entrada para evitar la contracción indeseable de la salida. Además, se asegura de que todos los píxeles se utilizan igualmente con frecuencia. Típicamente elegimos el padding simétrico en ambos lados de la altura de entrada y la anchura. En este caso nos referimos al padding $(p_\textrm{h}, p_\textrm{w})$. Lo más común es establecer $p_\textrm{h} = p_\textrm{w}$, en cuyo caso simplemente declaramos que elegimos el padding $p$.

Una convención similar se aplica a los pasos. Cuando el paso horizontal $s_\textrm{h}$ y el paso vertical $s_\textrm{w}$ coinciden, simplemente hablamos del paso $s$. El paso puede reducir la resolución de la salida, por ejemplo reduciendo la altura y el ancho de la salida a sólo $1/n$ de la altura y el ancho de la entrada para $n > 1$. Por defecto, el relleno es 0 y el paso es 1.

Hasta ahora todo el relleno que discutimos imágenes simplemente extendidas con ceros. Esto tiene un beneficio computacional significativo ya que es trivial de lograr. Además, los operadores pueden ser diseñados para aprovechar este relleno implícitamente sin la necesidad de asignar memoria adicional. Al mismo tiempo, permite a CNNs codificar la información de posición implícita dentro de una imagen, simplemente aprendiendo dónde está el "espacio en blanco". Hay muchas alternativas a cero-padding. [Alsallakh.Kokhlikyan.Miglani.ea.2020](https://d2l.ai/chapter_references/zreferences.html) proporcionó una visión general extensa de ellos (aunque sin un caso claro para cuándo utilizar rellenos no cero a menos que ocurran artefactos).

## Ejercicios
1. Dado el ejemplo de código final en esta sección con el tamaño del núcleo $(3, 5)$, el relleno $(0, 1)$ y el paso $(3, 4)$, calcule la forma de salida para comprobar si es consistente con el resultado experimental.
1. Para las señales de audio, ¿a qué corresponde un paso de 2?
1. Implementar el padding de espejo, es decir, el padding donde los valores de borde simplemente se reflejan para extender tensores.
1. ¿Cuáles son los beneficios computacionales de un paso mayor que 1?
1. ¿Cuáles podrían ser los beneficios estadísticos de un paso mayor que 1?
1. ¿Cómo implementarías un paso de $\frac{1}{2}$? ¿A qué corresponde? ¿Cuándo sería útil?


[Debate del original](https://discuss.d2l.ai/t/68)
